# 01 - Data, Baseline and First Explainability Maps

This notebook establishes the experimental substrate for the explainability study. It consolidates data loading, baseline availability and first attribution analysis into a single workflow.

Technical scope:

1. **Dataset interface validation**  
   Verify that AwA2 images, labels, class names and splits are loaded consistently through the manifest-based DataLoader.
2. **Baseline classifier availability**  
   Load or train the ResNet50 classifier used as the explained model in all later experiments.
3. **Initial attribution extraction**  
   Inspect Input Gradients, Grad-CAM, Score-CAM, Integrated Gradients and Expected Gradients on selected predictions.
4. **Gradient-level diagnostics**  
   Optionally compute tensor statistics for logits, gradients, activations and normalized saliency maps to verify that the maps are numerically meaningful.

The notebook is not a generic training notebook. Its purpose is to prepare and inspect the model that will later be stress-tested for explanation faithfulness.


In [ ]:
from pathlib import Path
from IPython.display import Image, display, HTML
import csv
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "scripts").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the project root containing src/ and scripts/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MANIFEST = PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv"
METADATA_ROOT = PROJECT_ROOT / "data" / "AWA2"
CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"

print("project_root:", PROJECT_ROOT)
print("manifest:", MANIFEST, "exists=", MANIFEST.exists())
print("checkpoint:", CHECKPOINT, "exists=", CHECKPOINT.exists())

# Set these flags to True only when you intentionally want to recompute outputs.
RUN_TRAINING = False
RUN_XAI = CHECKPOINT.exists()
RUN_GRADIENT_DIAGNOSTICS = False


## 1. Dataset and Manifest

A **manifest** is the CSV file that tells the project where every image is and which label it belongs to. Instead of scanning directories every time, the DataLoader reads this table.

Expected manifest columns:

- `filepath`: image location
- `label`: integer class id
- `class_name`: human-readable class name
- `split`: `train`, `val` or `test`

The DataLoader applies the standard ResNet/ImageNet preprocessing pipeline:

```text
Resize / crop -> convert to tensor -> normalize with ImageNet mean and std
```

This matters because ResNet50 was originally trained on ImageNet. If normalization is wrong, predictions and gradients become hard to interpret.


In [ ]:
from src.data import build_dataloaders, infer_class_map_path, denormalize_batch

CLASS_MAP = infer_class_map_path(MANIFEST)
loaders = build_dataloaders(
    manifest_path=MANIFEST,
    class_map_path=CLASS_MAP,
    batch_size=8,
    num_workers=0,
    pin_memory=False,
)

for split, loader in loaders.items():
    dataset = loader.dataset
    print(f"{split}: {len(dataset)} images | visible_classes={len(dataset.visible_classes)} | mapped_classes={len(dataset.classes)}")

images, labels, class_names, paths = next(iter(loaders['train']))
print('batch tensor shape:', images.shape)
print('label tensor shape:', labels.shape)
print('first class names:', list(class_names[:5]))
print('first image path:', paths[0])


### How to read this output

A correct batch should have shape similar to:

```text
[batch_size, 3, 224, 224]
```

That means each image has 3 RGB channels and has been transformed to the `224 x 224` input expected by ResNet50. The number of mapped classes should match the subset currently used by the project.


## 2. Baseline Model

The baseline classifier is **ResNet50**, loaded from `torchvision` and adapted to the AwA2 classes by replacing the final linear layer.

Conceptually:

```text
image -> ResNet convolutional backbone -> feature vector -> linear classifier -> animal class
```

The checkpoint stored in `outputs/checkpoints/best_resnet50_awa2.pt` is used by all later phases. XAI maps, stress tests, TCAV and Concept Bottleneck comparisons only make sense if this baseline exists.


In [ ]:
if RUN_TRAINING:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'training' / 'train_baseline.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint-output', str(CHECKPOINT),
        '--epochs', '5',
        '--batch-size', '32',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('training skipped')
    print('checkpoint exists:', CHECKPOINT.exists())


## 3. Training History

This cell reads the saved training history. In the current run, validation accuracy peaks early and then slightly decreases while training accuracy keeps increasing. That pattern suggests mild overfitting pressure, which is useful to mention in the final report.


In [ ]:
HISTORY = PROJECT_ROOT / 'outputs' / 'reports' / 'training_history.csv'
if HISTORY.exists():
    with HISTORY.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        print(row)
else:
    print('missing:', HISTORY)


### Interpretation checkpoint

Use the best validation epoch as the most credible baseline. If the model overfits, saliency maps can become even more dataset-specific, which strengthens the motivation for stress tests.


## Explainability Framing

A saliency map is a scalar field over the input image. For an image tensor `x` and target class score `S_c(x)`, the most direct attribution is the magnitude of the input derivative:

```text
saliency(x, c) = |d S_c(x) / d x|
```

This quantity measures **local sensitivity**, not causal necessity. A high value indicates that a small pixel change would affect the score locally; it does not prove that the model semantically used that region.

The first XAI extraction compares five mechanisms:

- **Input Gradients**: direct local derivative with respect to normalized input pixels through Captum Saliency.
- **Grad-CAM**: Captum LayerGradCam on the final convolutional activations.
- **Score-CAM**: local gradient-free CAM implementation kept for report parity.
- **Integrated Gradients**: accumulated gradients from a single blurred baseline image to the real image.
- **Expected Gradients**: Captum GradientShap over a distribution of reference baselines.

The important comparison is methodological: Grad-CAM and Score-CAM operate on convolutional feature maps, while Integrated Gradients and Expected Gradients operate in input space. Score-CAM removes gradient weighting from CAM; Expected Gradients reduces the dependence on one single baseline.


## 4. Attribution Method Comparison

This section generates a direct visual comparison between the maintained attribution methods.

- **Grad-CAM** uses gradients only to weight late convolutional channels. The map is coarse because `layer4[-1]` has low spatial resolution.
- **Score-CAM** does not use gradients. It masks the input with activation maps and measures how much each masked image supports the target class. This is closer to an intervention on feature maps, but it is computationally more expensive.
- **Integrated Gradients** integrates gradients along a path from one blurred baseline to the image. It explains score difference relative to that chosen baseline.
- **Expected Gradients** averages the same idea over multiple reference baselines, reducing dependence on a single baseline.

The output figure should be read column-wise: if two methods highlight different regions for the same image and same target class, they are answering different attribution questions rather than producing interchangeable explanations.


In [ ]:
XAI_FIGURE = PROJECT_ROOT / 'outputs' / 'figures' / 'xai_method_comparison_notebook.png'

if RUN_XAI:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'experiments' / 'run_xai.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint', str(CHECKPOINT),
        '--output', str(XAI_FIGURE),
        '--max-images', '4',
        '--ig-steps', '16',
        '--ig-internal-batch-size', '4',
        '--expected-gradient-samples', '12',
        '--expected-gradient-baselines', '16',
        '--scorecam-max-channels', '48',
        '--scorecam-batch-size', '16',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('XAI generation skipped; set RUN_XAI=True to recompute the method comparison.')


In [ ]:
print(XAI_FIGURE)
if XAI_FIGURE.exists():
    display(Image(filename=str(XAI_FIGURE)))
else:
    print('missing; run the previous cell to generate the comparison figure')


## Optional Attribution Tensor Diagnostics

This diagnostic cell recomputes explanations on a small batch and prints tensor statistics. It is disabled by default because Score-CAM, Integrated Gradients and Expected Gradients require multiple forward or backward evaluations.

The relevant checks are:

- logits and probabilities should be finite and class-specific;
- raw input gradients should contain both positive and negative values;
- normalized saliency maps should lie in `[0, 1]`;
- Grad-CAM activations should have spatial shape near `7 x 7` for ResNet50 layer4;
- Score-CAM weights should be valid target-class probabilities from masked inputs;
- Integrated Gradients attributions should be inspected before and after reduction to a single-channel map;
- Expected Gradients should log the sampled baseline pool and attributions.


In [ ]:
if RUN_GRADIENT_DIAGNOSTICS:
    import torch
    from src.model import build_resnet50_classifier, get_device
    from src.xai import (
        ScoreCAM,
        expected_gradients_maps,
        gradcam_saliency,
        input_gradient_saliency,
        integrated_gradients_maps,
        log_tensor_stats,
    )

    device = get_device('auto')
    num_classes = len(loaders['train'].dataset.classes)
    model = build_resnet50_classifier(num_classes=num_classes, pretrained=False)
    checkpoint = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
    model.to(device).eval()

    batch = next(iter(loaders['test']))
    reference_batch = next(iter(loaders['train']))
    images = batch[0][:2].to(device)
    labels = torch.as_tensor(batch[1][:2], dtype=torch.long, device=device)
    reference_images = reference_batch[0][:8].to(device)
    names = list(batch[2][:2])

    with torch.no_grad():
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        confidences, predictions = probabilities.max(dim=1)

    print('selected examples')
    for i, name in enumerate(names):
        print(f'[{i}] true={name} target={int(labels[i])} pred={int(predictions[i])} confidence={float(confidences[i]):.4f}')

    log_tensor_stats('diagnostics.images.normalized', images)
    log_tensor_stats('diagnostics.logits', logits)
    log_tensor_stats('diagnostics.probabilities', probabilities)

    input_maps = input_gradient_saliency(model, images, labels)
    target_layer = model.layer4[-1]
    gradcam_maps = gradcam_saliency(model, images, labels, target_layer)
    scorecam = ScoreCAM(model, target_layer, max_channels=16, batch_size=8)
    try:
        scorecam_maps = scorecam(images, labels)
    finally:
        scorecam.close()
    ig_maps, ig_attributions, ig_baseline = integrated_gradients_maps(
        model, images, labels, steps=8, internal_batch_size=2
    )
    eg_maps, eg_attributions, eg_baselines = expected_gradients_maps(
        model, images, labels, baselines=reference_images, n_samples=8, internal_batch_size=4
    )

    log_tensor_stats('diagnostics.input_gradient_maps', input_maps)
    log_tensor_stats('diagnostics.gradcam_maps', gradcam_maps)
    log_tensor_stats('diagnostics.scorecam_maps', scorecam_maps)
    log_tensor_stats('diagnostics.ig_maps', ig_maps)
    log_tensor_stats('diagnostics.ig_attributions', ig_attributions)
    log_tensor_stats('diagnostics.ig_blurred_baseline', ig_baseline)
    log_tensor_stats('diagnostics.expected_gradients_maps', eg_maps)
    log_tensor_stats('diagnostics.expected_gradients_attributions', eg_attributions)
    log_tensor_stats('diagnostics.expected_gradients_baselines', eg_baselines)
else:
    print('attribution diagnostics skipped; set RUN_GRADIENT_DIAGNOSTICS=True to execute')
